# 02 - Customer Feature Engineering
Nội dung chính:
- Join 3 bảng: `sales`, `product`, `customer`.
- Tạo feature cơ bản và nâng cao theo `customer_id`.
- Gộp thông tin nhân khẩu học: giới tính, khu vực, hạng thành viên.
- Lưu output dùng cho bước modeling.


## 1) Setup


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
print('Project root:', PROJECT_ROOT)
print('Processed data dir:', DATA_PROCESSED)


In [ ]:
from build_customer_features import (
    load_clean_data,
    build_merged_transactions,
    build_customer_features,
    save_outputs,
)

sales, customer, product = load_clean_data(DATA_PROCESSED)

print('sales shape   :', sales.shape)
print('customer shape:', customer.shape)
print('product shape :', product.shape)


## 2) Join 3 bảng thành bảng giao dịch gộp


In [ ]:
merged = build_merged_transactions(sales, customer, product)
print('merged shape:', merged.shape)
merged.head()


In [ ]:
merge_quality = pd.DataFrame({
    'metric': [
        'unique customers in merged',
        'missing category',
        'missing gender',
        'missing customer_region',
        'missing loyalty_tier',
    ],
    'value': [
        merged['customer_id'].nunique(),
        int(merged['category'].isna().sum()),
        int(merged['gender'].isna().sum()),
        int(merged['customer_region'].isna().sum()),
        int(merged['loyalty_tier'].isna().sum()),
    ]
})
merge_quality


## 3) Tạo customer-level features

C?c feature ch?nh:
- `total_spent`, `avg_order_value`, `total_quantity`, `num_purchases`
- `days_since_first_purchase`, `days_since_last_purchase`
- `purchase_frequency`, `preferred_category`, `has_discount`
- `gender`, `customer_region`, `loyalty_tier`


In [ ]:
customer_features = build_customer_features(merged, customer)
print('customer_features shape:', customer_features.shape)
customer_features.head()


In [ ]:
required_columns = [
    'customer_id',
    'total_spent',
    'avg_order_value',
    'total_quantity',
    'num_purchases',
    'days_since_first_purchase',
    'days_since_last_purchase',
    'purchase_frequency',
    'preferred_category',
    'has_discount',
    'gender',
    'customer_region',
    'loyalty_tier',
]

missing_cols = [c for c in required_columns if c not in customer_features.columns]
print('Missing required columns:', missing_cols)
print('Total missing values in required columns:', int(customer_features[required_columns].isna().sum().sum()))


## 4) Lưu output


In [ ]:
save_outputs(merged, customer_features, DATA_PROCESSED)

merged_path = DATA_PROCESSED / 'merged_transactions.csv'
features_path = DATA_PROCESSED / 'customer_features.csv'

print('Saved:', merged_path, '| rows =', sum(1 for _ in open(merged_path, 'r', encoding='utf-8')) - 1)
print('Saved:', features_path, '| rows =', sum(1 for _ in open(features_path, 'r', encoding='utf-8')) - 1)


## 5) Biểu dồ cho báo cáo


In [ ]:
top_categories = customer_features['preferred_category'].value_counts().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
top_categories.plot(kind='bar')
plt.title('Preferred Category Distribution')
plt.xlabel('Category')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
customer_features['total_spent'].plot(kind='hist', bins=25)
plt.title('Total Spent Distribution')
plt.xlabel('Total Spent')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


## 6) Lưu ý khi trình bày

Trong bộ dữ liệu hiện tại, `order_date` chỉ có một mốc thời gian (`2025-07-06`) nên:
- `days_since_first_purchase` v? `days_since_last_purchase` bằng 0 cho mội khách hàng.
- `purchase_frequency` cũng bằng 0 vì không có khoảng cách ngày giữa các lần mua.

Khi có dữ liệu đa thời điểm, 3 feature thời gian này sẽ phản ánh hành vi mua tốt hơn.
